In [1]:
cd ..

/Users/smrutichourasia/Documents/GitHub/Project/schema_field_map


In [2]:
pwd

'/Users/smrutichourasia/Documents/GitHub/Project/schema_field_map'

# Setup LLM Manager Test

In [6]:
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv(override=True)  # reads .env in project root
from src.clients import CortexSettings, LLMManager
settings = CortexSettings()

In [7]:
class SimpleTest(BaseModel):
    answer: str = Field(description="A short answer")
    confidence: float = Field(ge=0.0, le=1.0)

manager = LLMManager(settings)

result = manager.call_routing(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"},
    ],
    response_model=SimpleTest,
)
print(f"Answer: {result.answer}")
print(f"Confidence: {result.confidence}")

InstructorRetryException: <failed_attempts>

<generation number="1">
<exception>
    Connection error.
</exception>
<completion>
    None
</completion>
</generation>

<generation number="2">
<exception>
    Connection error.
</exception>
<completion>
    None
</completion>
</generation>

<generation number="3">
<exception>
    Connection error.
</exception>
<completion>
    None
</completion>
</generation>

</failed_attempts>

<last_exception>
    Connection error.
</last_exception>

# Setup Embedding Test

In [9]:

# Cell 1 — Load schemas into Pydantic models
import json
from src.models import SourceField, DestinationField

with open("mappings/mysql/schema_clean.json") as f:
    mysql_data = json.load(f)

with open("mappings/mongodb/schema_clean.json") as f:
    mongo_data = json.load(f)

source_fields = []
for table_name, fields in mysql_data["tables"].items():
    for field_dict in fields:
        sf = SourceField(table_name=table_name, **field_dict)
        source_fields.append(sf)

dest_fields = []
for coll_name, fields in mongo_data["collections"].items():
    for field_dict in fields:
        df = DestinationField(**field_dict)
        dest_fields.append(df)

print(f"Source fields: {len(source_fields)}")
print(f"Destination fields: {len(dest_fields)}")

Source fields: 34
Destination fields: 40


In [ ]:
# Cell 2 — Initialize embedding manager and embed destinations
from src.nodes.embedding_manager import EmbeddingManager
em = EmbeddingManager()

text_reprs = [df.text_repr for df in dest_fields]
paths = [df.path for df in dest_fields]
collection_names = [df.collection_name for df in dest_fields]

matrix = em.embed_destination_fields(text_reprs, paths)
print(f"Destination matrix: {matrix.shape}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7299.25it/s]


Destination matrix: (40, 384)


In [13]:

# Cell 3 — Test retrieval for individual source fields
test_fields = [
    ("rec_stat", "employees"),
    ("f_name", "employees"),
    ("is_remote", "employees"),
    ("dept_cd", "departments"),
    ("loc_nm", "locations"),
]

for field_name, collection in test_fields:
    sf = next(f for f in source_fields if f.field_name == field_name)
    candidates = em.retrieve_candidates(
        source_text_repr=sf.text_repr,
        source_field_name=sf.field_name,
        collection_filter=collection,
        collection_names=collection_names,
    )
    print(f"\n{sf.table_name}.{sf.field_name}:")
    if candidates:
        for c in candidates:
            print(f"  → {c['destination_field']:30s} hybrid={c['hybrid_score']:.4f} ({c['retrieval_confidence_prior']})")
    else:
        print("  → No candidates above threshold")



emp_master.rec_stat:
  → employment.status              hybrid=0.4924 (LOW)
  → employment.endDate             hybrid=0.4179 (LOW)

emp_master.f_name:
  → No candidates above threshold

emp_master.is_remote:
  → No candidates above threshold

dept_info.dept_cd:
  → code                           hybrid=0.4017 (LOW)

locations.loc_nm:
  → No candidates above threshold


In [ ]:
# Cell 4 — Show impact of enriched text_repr (simulating semantic profiling)
# This demonstrates WHY build_semantic_profiles matters

# Without enrichment
raw = "table:emp_master | field:f_name | type:VARCHAR(50) | nullable:False"
candidates_raw = em.retrieve_candidates(
    source_text_repr=raw,
    source_field_name="f_name",
    collection_filter="employees",
    collection_names=collection_names,
)
print("f_name (raw text_repr):")
for c in candidates_raw:
    print(f"  → {c['destination_field']:30s} hybrid={c['hybrid_score']:.4f}")

# With enrichment (simulating what build_semantic_profiles would add)
enriched = (
    "table:emp_master | field:f_name | type:VARCHAR(50) | nullable:False | "
    "business_meaning:Employee first name | keywords:first name, given name, firstName"
)
candidates_enriched = em.retrieve_candidates(
    source_text_repr=enriched,
    source_field_name="f_name",
    collection_filter="employees",
    collection_names=collection_names,
)
print("\nf_name (enriched text_repr):")
for c in candidates_enriched:
    print(f"  → {c['destination_field']:30s} hybrid={c['hybrid_score']:.4f}")

